In [1]:
import importlib, subprocess, sys
if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', '-q'])

import os, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
os.makedirs('./models', exist_ok=True)
print(f'XGBoost {xgb.__version__} ready')

XGBoost 2.1.3 ready


In [2]:
# ─── Only change DATA_PATH if your file name is different ───
DATA_PATH  = './data_sets.xlsx'
SHEET_NAME = 'Monthly_Data'
MODEL_DIR  = './models'

# Nepal 6-Season Calendar (Ritu)
SEASON_NAME = {
     1:'Shishir (Winter)',  2:'Shishir (Winter)',
     3:'Basanta (Spring)',  4:'Basanta (Spring)',
     5:'Grishma (Summer)',  6:'Grishma (Summer)',
     7:'Barsha (Monsoon)',  8:'Barsha (Monsoon)',
     9:'Sharad (Autumn)',  10:'Sharad (Autumn)',
    11:'Hemanta (Pre-Winter)', 12:'Hemanta (Pre-Winter)',
}
SEASON_ENC = {
    'Shishir (Winter)':0, 'Basanta (Spring)':1,
    'Grishma (Summer)':2, 'Barsha (Monsoon)':3,
    'Sharad (Autumn)':4,  'Hemanta (Pre-Winter)':5,
}
DISC_ENC  = {'None':0, 'Low':1, 'Medium':2, 'High':3}
DISC_RATE = {'None':0.0, 'Low':0.03, 'Medium':0.05, 'High':0.10}

In [3]:
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df_raw['Discount Band'] = df_raw['Discount Band'].fillna('None')

ALL_PRODUCTS  = sorted(df_raw['Product'].unique())
ALL_SEGMENTS  = sorted(df_raw['Segment'].unique())
BASE_YEAR     = int(df_raw['Year'].min())
SEG_ENC       = {s:i for i,s in enumerate(ALL_SEGMENTS)}
PROD_MAP      = {p:i for i,p in enumerate(ALL_PRODUCTS)}
ACTIVE_MONTHS = df_raw.groupby('Product')['Month Number'].apply(lambda x: sorted(x.unique().tolist())).to_dict()
PRODUCT_SPECS = df_raw.sort_values(['Year','Month Number']).groupby('Product').last()[['Manufacturing Price','Sale Price','Segment']].to_dict('index')
PRODUCT_DISC  = df_raw.groupby('Product')['Discount Band'].agg(lambda x: x.mode()[0]).to_dict()

print(f'Loaded {len(df_raw):,} rows | {len(ALL_PRODUCTS)} products | years {BASE_YEAR}–{df_raw["Year"].max()}')

Loaded 2,121 rows | 31 products | years 2015–2026


In [4]:
def build_features(df):
    df = df.copy()
    df['Discount Band'] = df['Discount Band'].fillna('None')
    df['Date'] = pd.to_datetime(df['Year'].astype(str)+'-'+df['Month Number'].astype(str).str.zfill(2)+'-01')
    df = df.sort_values(['Product','Date']).reset_index(drop=True)
    df['Month']       = df['Date'].dt.month
    df['Year_num']    = df['Date'].dt.year
    df['Quarter']     = df['Date'].dt.quarter
    df['Month_sin']   = np.sin(2*np.pi*df['Month']/12)
    df['Month_cos']   = np.cos(2*np.pi*df['Month']/12)
    df['Qtr_sin']     = np.sin(2*np.pi*df['Quarter']/4)
    df['Qtr_cos']     = np.cos(2*np.pi*df['Quarter']/4)
    df['Season_enc']  = df['Month'].map(SEASON_NAME).map(SEASON_ENC)
    df['Year_trend']  = df['Year_num'] - BASE_YEAR
    df['Segment_enc'] = df['Segment'].map(SEG_ENC)
    df['Product_enc'] = df['Product'].map(PROD_MAP)
    df['DiscBand_enc']= df['Discount Band'].map(DISC_ENC).fillna(0).astype(int)
    for lag in [1,2,3,6,12]:
        df[f'Sales_lag{lag}']  = df.groupby('Product')['Sales'].shift(lag)
        df[f'Profit_lag{lag}'] = df.groupby('Product')['Profit'].shift(lag)
    for win in [3,6]:
        df[f'Sales_roll{win}_mean']  = df.groupby('Product')['Sales'].transform(lambda x: x.shift(1).rolling(win,min_periods=1).mean())
        df[f'Sales_roll{win}_std']   = df.groupby('Product')['Sales'].transform(lambda x: x.shift(1).rolling(win,min_periods=2).std().fillna(0))
        df[f'Profit_roll{win}_mean'] = df.groupby('Product')['Profit'].transform(lambda x: x.shift(1).rolling(win,min_periods=1).mean())
        df[f'Profit_roll{win}_std']  = df.groupby('Product')['Profit'].transform(lambda x: x.shift(1).rolling(win,min_periods=2).std().fillna(0))
    df['Discount_Rate']      = np.where(df['Gross Sales']!=0, df['Discounts']/df['Gross Sales'],0)
    df['Revenue_per_Unit']   = np.where(df['Units Sold']!=0,  df['Sales']/df['Units Sold'],0)
    df['Cost_per_Unit']      = np.where(df['Units Sold']!=0,  df['COGS']/df['Units Sold'],0)
    df['Price_to_ManufCost'] = np.where(df['Manufacturing Price']!=0, df['Sale Price']/df['Manufacturing Price'],0)
    df['Gross_Margin_pct']   = np.where(df['Gross Sales']!=0, (df['Gross Sales']-df['COGS'])/df['Gross Sales'],0)
    return df

DROP_COLS    = ['Sales','Profit','Date','Segment','Product','Discount Band','Month Number','Year']
df_feat      = build_features(df_raw)
FEATURE_COLS = [c for c in df_feat.columns if c not in DROP_COLS]
LAG_COLS     = [c for c in FEATURE_COLS if 'lag' in c or 'roll' in c]
df_model     = df_feat.dropna(subset=LAG_COLS).reset_index(drop=True)
split_idx    = int(len(df_model)*0.80)
df_train     = df_model.iloc[:split_idx]
df_test      = df_model.iloc[split_idx:]
print(f'Features: {len(FEATURE_COLS)} | Train: {len(df_train):,} | Test: {len(df_test):,}')

Features: 41 | Train: 1,399 | Test: 350


In [5]:
XGB_PARAMS = dict(
    n_estimators=800, learning_rate=0.03, max_depth=6,
    subsample=0.80, colsample_bytree=0.80,
    reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5, gamma=0.1,
    random_state=42, tree_method='hist',
    early_stopping_rounds=40, eval_metric='rmse', verbosity=0,
)

for tgt in ['Sales','Profit']:
    print(f'Training {tgt} model...')
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(df_train[FEATURE_COLS], df_train[tgt],
              eval_set=[(df_test[FEATURE_COLS], df_test[tgt])], verbose=False)
    preds = model.predict(df_test[FEATURE_COLS])
    r2    = r2_score(df_test[tgt], preds)
    mape  = np.mean(np.abs((df_test[tgt]-preds)/np.where(df_test[tgt]==0,1,df_test[tgt])))*100
    print(f'  R2={r2:.4f}  MAPE={mape:.2f}%  ({"Excellent" if r2>0.95 else "Good" if r2>0.85 else "Review"})')
    model.save_model(f'{MODEL_DIR}/{tgt.lower()}_xgb.json')
    joblib.dump(model, f'{MODEL_DIR}/{tgt.lower()}_xgb.pkl')
    print(f'  Saved: models/{tgt.lower()}_xgb.json + .pkl')

# Save everything predict.ipynb needs
for name, obj in [
    ('feature_cols', FEATURE_COLS), ('prod_map', PROD_MAP),
    ('seg_enc', SEG_ENC),           ('active_months', ACTIVE_MONTHS),
    ('product_specs', PRODUCT_SPECS),('product_disc', PRODUCT_DISC),
    ('base_year', BASE_YEAR),       ('season_name', SEASON_NAME),
    ('season_enc', SEASON_ENC),     ('all_products', ALL_PRODUCTS),
    ('disc_enc', DISC_ENC),         ('disc_rate', DISC_RATE),
    ('df_raw', df_raw),
]:
    joblib.dump(obj, f'{MODEL_DIR}/{name}.pkl')

print()
print('='*50)
print('  TRAINING COMPLETE — MODEL SAVED')
print('  Open predict.ipynb to get predictions.')
print('  You never need to run this file again.')
print('='*50)

Training Sales model...
  R2=0.9950  MAPE=2.11%  (Excellent)
  Saved: models/sales_xgb.json + .pkl
Training Profit model...
  R2=0.9754  MAPE=4.51%  (Excellent)
  Saved: models/profit_xgb.json + .pkl

  TRAINING COMPLETE — MODEL SAVED
  Open predict.ipynb to get predictions.
  You never need to run this file again.
